# Notebook 1: Data Landscape Overview

Bird's-eye view of the cleaned PPD dataset (4,000 rows × 291 columns, FY2001–2021).

**Sections:**
1. Column inventory by thematic category
2. Missingness heatmap (column × fiscal year)
3. Summary statistics for all numeric columns
4. Distribution gallery (top ~20 variables)
5. Correlation matrix (key ~30 variables)
6. Plan-level vs. system-level structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

OUTPUT_DIR = Path('../../output_python/exploration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load cleaned plan-level data
df = pd.read_csv('../../output_python/ppd_plan_level_clean.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Fiscal years: {df["fy"].min():.0f}–{df["fy"].max():.0f}')
print(f'Unique plans (ppd_id): {df["ppd_id"].nunique()}')
print(f'Unique systems (pub_id): {df["pub_id"].nunique()}')

## Section 1: Column Inventory

Group all 291 columns into thematic categories with dtype, non-null count, and example values.

In [ ]:
# Define thematic column groups
COLUMN_GROUPS = {
    'Identifiers': [
        'ppd_id', 'PlanName', 'fy', 'system_id', 'PlanFullName', 'source_PlanBasics',
        'ppd_name', 'pub_id', 'pub_system_name', 'planname',
        'preqin_id', 'preqin_name',
        'tr13fid1', 'tr13fname1', 'tr13fid2', 'tr13fname2',
        'tr13fid3', 'tr13fname3', 'tr13fid4', 'tr13fname4',
    ],
    'Plan Characteristics': [
        'FiscalYearType', 'PlanInceptionYear', 'PlanClosed', 'PlanYearClosed',
        'AdministeringGovt', 'StateAbbrev', 'StateName', 'GovtName',
        'PlanType', 'EmployeeTypeCovered', 'SocSecCovered', 'SocSecCovered_verbatim',
        'CostStructure', 'EmployerType', 'CostSharing', 'BenefitsWebsite',
        'EEGroupID', 'TierID', 'ActRptDate', 'fye',
        'region', 'division', 'ppd_zipcode',
    ],
    'Actuarial Assumptions & Methods': [
        'ActCostMeth_GASB', 'AssetValMeth_GASB', 'FundingMeth_GASB',
        'InflationAssumption_GASB', 'InvestmentReturnAssumption_GASB',
        'ActCostMethCode_GASB', 'AssetValMethCode_GASB', 'AssetSmoothingPeriod_GASB',
        'FundingMethCode1_GASB', 'FundingMethCode2_GASB', 'UAALAmortPeriod_GASB',
        'BlendedDiscountRate',
        'AssetValMeth', 'PhaseIn', 'AssetValMeth_note',
        'ActCostMeth', 'ActCostMeth_note', 'FundingMeth', 'FundingMeth_note',
        'AssetValMethCode', 'SmoothingReset', 'GainlossConcept',
        'GainLossBase_1', 'GainLossBase_2', 'GainLoss', 'GainLossPeriod',
        'PhaseInPercent', 'PhaseInPeriods', 'PhaseInType',
        'GainLossRecognition', 'AssetSmoothingBaseline', 'ExpectedReturnMethod',
        'AddSubtractGainLoss', 'UpperCorridor', 'LowerCorridor',
        'ActCostMethCode', 'FundMethCode_1', 'FundMethCode_2',
        'PayrollGrowthAssumption', 'TotAmortPeriod', 'RemainingAmortPeriod',
        'UAALYearEstablished', 'WageInflation',
    ],
    'Funding & Liabilities': [
        'ActAssets_GASB', 'ActLiabilities_GASB', 'ActFundedRatio_GASB', 'UAAL_GASB',
        'ActLiabilities_other', 'payroll', 'RequiredContribution',
        'PercentReqContPaid', 'TotalPensionLiability', 'NetPosition',
        'NetPensionLiability', 'ActAssets_est', 'ActLiabilities_est',
        'ActFundedRatio_est', 'RequiredContribution_est', 'ActFundedRatio_GASB67',
        'MktAssets_Smooth', 'ActAssets_Smooth', 'NetFlows_smooth',
        'MktAssets_ActRpt', 'ActAssets_AVA',
        'ActLiabilities_EAN', 'ActLiabilities_PUC',
    ],
    'Contributions & Normal Cost': [
        'contrib_EE_regular', 'contrib_ER_regular', 'contrib_ER_state',
        'contrib_EE_PurchaseService', 'contrib_EE_other', 'contrib_ER_other',
        'contrib_other', 'POB_Flag', 'POB_Amount', 'contrib_tot',
        'contrib_ER_tot', 'contrib_other_tot', 'ContributionFY',
        'NormCostRate_tot', 'NormCostRate_EE', 'NormCostRate_ER',
        'ReqContRate_ER', 'ReqContRate_tot', 'ReqContRate_ER_Stat',
        'ReqContRate_tot_Stat',
        'NormCostAmount_tot', 'NormCostAmount_EE', 'NormCostAmount_ER',
        'ReqContAmount_ER', 'ReqContAmount_tot',
        'ReqContAmount_ER_Stat', 'ReqContAmount_tot_Stat',
        'NormCostRate_tot_est', 'NormCostRate_EE_est', 'NormCostRate_ER_est',
        'ReqContRate_ER_est', 'ProjectedPayroll', 'UAALRate',
    ],
    'Investment Returns': [
        'InvestmentConsultantName', 'InvestmentConsultantCode',
        'InvestmentReturn_1yr', 'InvestmentReturn_2yr', 'InvestmentReturn_3yr',
        'InvestmentReturn_4yr', 'InvestmentReturn_5yr', 'InvestmentReturn_7yr',
        'InvestmentReturn_8yr', 'InvestmentReturn_10yr', 'InvestmentReturn_12yr',
        'InvestmentReturn_15yr', 'InvestmentReturn_20yr', 'InvestmentReturn_25yr',
        'InvestmentReturn_30yr', 'InvestmentReturn_LongTerm', 'InvestmentReturn_LTStartYear',
        'GrossReturns', 'GeoReturn_est', 'GeoGrowth_est',
        'InvestmentReturn_1yr_est', 'InvestmentReturn_5yr_est',
        'InvestmentReturn_10yr_est', 'AvgReturn_3yr', 'AvgReturn_5yr', 'AvgReturn_10yr',
        'ret_bgnassets',
    ],
    'Investment Income & Expenses': [
        'FairValueChange_investments', 'FairValueChange_RealEstate',
        'income_interest', 'income_dividends', 'income_InterestAndDividends',
        'income_RealEstate', 'income_PrivateEquity', 'income_alternatives',
        'income_international', 'income_OtherInvestments',
        'expense_RealEstate', 'expense_PrivateEquity', 'expense_alternatives',
        'expense_OtherInvestments', 'expense_investments',
        'FVChange_SecLend', 'income_SecuritiesLending', 'expense_SecuritiesLending',
        'income_SecuritiesLendingRebate', 'FVChange_SecLend_UG',
        'expense_SecLendMgmtFees',
        'income_OtherAdditions', 'income_net',
        'FairValueChange_tot', 'income_interest_dividends_tot',
        'expense_investments_tot', 'SecLend_tot', 'investments_net',
    ],
    'Benefits & Expenses': [
        'expense_TotBenefits', 'expense_RetBenefits', 'expense_DisabilityBenefits',
        'expense_DeathBenefits', 'expense_DROPBenefits', 'expense_SurvivorBenefits',
        'expense_COLABenefits', 'expense_LumpSumBenefits', 'expense_OtherBenefits',
        'expense_refunds', 'expense_AdminExpenses', 'expense_Depreciation',
        'expense_OtherDeductions', 'expense_net', 'expense_other_tot',
        'adjustment_MktAssets', 'MktAssets_net', 'BegMktAssets_net',
    ],
    'Membership & Demographics': [
        'actives_tot', 'ActiveSalaries', 'ActiveAge_avg', 'ActiveTenure_avg',
        'ActiveSalary_avg', 'InactiveVestedMembers', 'InactiveNonVested',
        'beneficiaries_tot', 'benefits_tot', 'BeneficiaryAge_avg',
        'BeneficiaryBenefit_avg', 'beneficiaries_ServiceRetirees',
        'benefits_ServiceRetirees', 'ServiceRetireeAge_avg',
        'ServiceRetireeBenefit_avg', 'ServiceRetAge_avg', 'ServiceRetTenure_avg',
        'beneficiaries_DisabilityRetirees', 'beneficiaries_DependentSurvivors',
        'benefits_DisabilityRetirees', 'beneficiaries_survivors',
        'beneficiaries_SpousalSurvivors', 'beneficiaries_other',
        'DROPMembers', 'OtherMembers', 'TotMembership',
        'ActiveSalary_avg_est', 'BeneficiaryBenefit_avg_est',
    ],
    'Present Value of Future Benefits': [
        'PVFB_InactiveNonVested', 'PVFB_active', 'PVFB_InactiveVested',
        'PVFB_retiree', 'PVFB_other', 'PVFB_tot',
        'PVFNC_tot', 'PVFNC_EE', 'PVFNC_ER', 'PVFS',
    ],
    'Asset Allocation — Target': [
        'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
        'HFTotal_Trgt', 'AltMiscTotal_Trgt', 'COMDTotal_Trgt',
        'CashTotal_Trgt', 'OtherTotal_Trgt', 'Leverage_Total_Trgt',
    ],
    'Asset Allocation — Actual': [
        'EQTotal_Actl', 'FITotal_Actl', 'RETotal_Actl', 'PETotal_Actl',
        'HFTotal_Actl', 'AltMiscTotal_Actl', 'COMDTotal_Actl',
        'CashTotal_Actl', 'OtherTotal_Actl', 'Leverage_Total_Actl', 'Leverage_Flag',
    ],
    'Asset Allocation — Returns': [
        'EQTotal_Rtrn', 'FITotal_Rtrn', 'RETotal_Rtrn', 'PETotal_Rtrn',
        'HFTotal_Rtrn', 'AltMiscTotal_Rtrn', 'COMDTotal_Rtrn',
        'CashTotal_Rtrn', 'OtherTotal_Rtrn', 'TotFund_BnchmrkRtrn',
    ],
    'Consultant & Flags': [
        'general_consultant', 'consultant_id', 'ActuarialFirm', 'ActuarialFirmCode',
        'trgt_zero_actl_nonzero', 'is_actual_replaced', 'is_target_replaced',
    ],
}

# Build inventory table
rows = []
for group, cols in COLUMN_GROUPS.items():
    for col in cols:
        if col in df.columns:
            non_null = df[col].notna().sum()
            pct = non_null / len(df) * 100
            example = df[col].dropna().iloc[0] if non_null > 0 else 'N/A'
            if isinstance(example, float):
                example = f'{example:.4g}'
            else:
                example = str(example)[:40]
            rows.append({
                'Group': group,
                'Column': col,
                'Dtype': str(df[col].dtype),
                'Non-Null': f'{non_null:,}',
                'Coverage': f'{pct:.0f}%',
                'Example': example,
            })

inventory = pd.DataFrame(rows)

# Check for any columns we missed
all_grouped = set()
for cols in COLUMN_GROUPS.values():
    all_grouped.update(cols)
ungrouped = [c for c in df.columns if c not in all_grouped]
if ungrouped:
    print(f'⚠ Ungrouped columns: {ungrouped}')

print(f'Column inventory: {len(inventory)} columns across {len(COLUMN_GROUPS)} groups\n')

# Summary by group
group_summary = inventory.groupby('Group').agg(
    Columns=('Column', 'count'),
).reindex(COLUMN_GROUPS.keys())
print(group_summary.to_string())

In [ ]:
# Show full inventory (scrollable)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', 50)
inventory

## Section 2: Missingness Heatmap

Visualize data availability by column group × fiscal year. Darker = more missing data.

In [ ]:
# Missingness by fiscal year for key numeric columns
key_cols = (
    # Allocation targets
    ['EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
     'HFTotal_Trgt', 'CashTotal_Trgt', 'AltMiscTotal_Trgt', 'COMDTotal_Trgt', 'OtherTotal_Trgt']
    # Allocation actuals
    + ['EQTotal_Actl', 'FITotal_Actl', 'RETotal_Actl', 'PETotal_Actl',
       'HFTotal_Actl', 'CashTotal_Actl']
    # Returns
    + ['InvestmentReturn_1yr', 'ret_bgnassets']
    # Funding
    + ['ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB', 'UAAL_GASB']
    # Membership
    + ['TotMembership', 'actives_tot', 'beneficiaries_tot', 'MktAssets_net']
    # Contributions
    + ['contrib_tot', 'payroll']
)

# Compute non-null fraction by fiscal year
miss_by_fy = []
for fy in sorted(df['fy'].unique()):
    sub = df[df['fy'] == fy]
    row = {'fy': int(fy)}
    for col in key_cols:
        row[col] = sub[col].notna().mean() * 100
    miss_by_fy.append(row)

miss_df = pd.DataFrame(miss_by_fy).set_index('fy')

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(miss_df.T, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=100, linewidths=0.5, ax=ax,
            cbar_kws={'label': '% Non-Missing'})
ax.set_title('Data Coverage by Variable × Fiscal Year (% non-missing)')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'missingness_heatmap.png')
plt.show()

## Section 3: Summary Statistics

Extended percentile table for all key numeric variables.

In [ ]:
# Comprehensive summary statistics for key numeric variables
summary_cols = [
    # Allocation targets
    'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
    'HFTotal_Trgt', 'AltMiscTotal_Trgt', 'COMDTotal_Trgt', 'CashTotal_Trgt', 'OtherTotal_Trgt',
    # Allocation actuals
    'EQTotal_Actl', 'FITotal_Actl', 'RETotal_Actl', 'PETotal_Actl',
    'HFTotal_Actl', 'AltMiscTotal_Actl', 'COMDTotal_Actl', 'CashTotal_Actl', 'OtherTotal_Actl',
    # Funding & actuarial
    'ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB', 'InflationAssumption_GASB',
    'UAAL_GASB', 'MktAssets_net', 'BegMktAssets_net',
    # Performance
    'InvestmentReturn_1yr', 'ret_bgnassets',
    # Membership
    'TotMembership', 'actives_tot', 'beneficiaries_tot',
    'ActiveAge_avg', 'ActiveTenure_avg', 'ActiveSalary_avg',
    'BeneficiaryAge_avg', 'BeneficiaryBenefit_avg',
    # Contributions
    'contrib_tot', 'payroll', 'PercentReqContPaid',
]

# Only include columns that exist
summary_cols = [c for c in summary_cols if c in df.columns]

stats = df[summary_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
stats['non_null'] = df[summary_cols].notna().sum()
stats['skew'] = df[summary_cols].skew()

# Reorder columns
stats = stats[['non_null', 'mean', 'std', 'min', '10%', '25%', '50%', '75%', '90%', 'max', 'skew']]
stats.columns = ['N', 'Mean', 'Std', 'Min', 'p10', 'p25', 'Median', 'p75', 'p90', 'Max', 'Skew']

# Flag highly skewed variables
stats['Highly Skewed'] = stats['Skew'].abs() > 2

print('Summary Statistics — Key Numeric Variables')
print('=' * 100)
stats.round(4)

## Section 4: Distribution Gallery

Histograms / KDEs for the ~20 most important variables.

In [ ]:
# Distribution gallery — grid of histograms with KDE overlay
dist_vars = [
    ('EQTotal_Trgt', 'Equity Target'),
    ('FITotal_Trgt', 'Fixed Income Target'),
    ('RETotal_Trgt', 'Real Estate Target'),
    ('PETotal_Trgt', 'Private Equity Target'),
    ('HFTotal_Trgt', 'Hedge Fund Target'),
    ('CashTotal_Trgt', 'Cash Target'),
    ('EQTotal_Actl', 'Equity Actual'),
    ('FITotal_Actl', 'Fixed Income Actual'),
    ('ActFundedRatio_GASB', 'GASB Funding Ratio'),
    ('InvestmentReturnAssumption_GASB', 'Assumed Return'),
    ('InvestmentReturn_1yr', '1-Yr Return (PPD)'),
    ('ret_bgnassets', 'Net Return (ret_bgnassets)'),
    ('MktAssets_net', 'Net Market Assets ($)'),
    ('TotMembership', 'Total Membership'),
    ('actives_tot', 'Active Members'),
    ('beneficiaries_tot', 'Beneficiaries'),
    ('ActiveAge_avg', 'Avg Active Age'),
    ('ActiveSalary_avg', 'Avg Active Salary ($)'),
    ('contrib_tot', 'Total Contributions ($)'),
    ('payroll', 'Payroll ($)'),
]

n_vars = len(dist_vars)
ncols = 4
nrows = (n_vars + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.2))
axes = axes.flatten()

for i, (col, label) in enumerate(dist_vars):
    ax = axes[i]
    vals = df[col].dropna()
    if len(vals) == 0:
        ax.set_title(f'{label}\n(no data)')
        continue
    
    # For dollar/count variables with huge ranges, use log scale
    use_log = col in ['MktAssets_net', 'TotMembership', 'actives_tot',
                       'beneficiaries_tot', 'contrib_tot', 'payroll']
    
    if use_log:
        vals_plot = vals[vals > 0]
        if len(vals_plot) > 0:
            ax.hist(np.log10(vals_plot), bins=40, color='#1f77b4', edgecolor='white', alpha=0.8)
            ax.set_xlabel(f'log10({col})')
    else:
        ax.hist(vals, bins=40, color='#1f77b4', edgecolor='white', alpha=0.8)
    
    ax.set_title(f'{label}\n(N={len(vals):,}, median={vals.median():.3g})', fontsize=9)
    ax.tick_params(labelsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribution Gallery — Key Variables', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'distribution_gallery.png')
plt.show()

## Section 5: Correlation Matrix

Pairwise correlations among ~30 key variables. Highlight |r| > 0.5.

In [ ]:
# Correlation matrix for key variables
corr_cols = [
    # Allocation targets
    'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
    'HFTotal_Trgt', 'CashTotal_Trgt',
    # Allocation actuals
    'EQTotal_Actl', 'FITotal_Actl', 'RETotal_Actl', 'PETotal_Actl',
    'HFTotal_Actl', 'CashTotal_Actl',
    # Funding
    'ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB',
    # Performance
    'InvestmentReturn_1yr', 'ret_bgnassets',
    # Membership
    'TotMembership', 'actives_tot', 'beneficiaries_tot',
    'ActiveAge_avg', 'ActiveSalary_avg', 'BeneficiaryBenefit_avg',
    # Financial
    'MktAssets_net', 'payroll', 'contrib_tot',
]

corr_cols = [c for c in corr_cols if c in df.columns]

# Short labels for readability
short_labels = {
    'EQTotal_Trgt': 'EQ Trgt', 'FITotal_Trgt': 'FI Trgt',
    'RETotal_Trgt': 'RE Trgt', 'PETotal_Trgt': 'PE Trgt',
    'HFTotal_Trgt': 'HF Trgt', 'CashTotal_Trgt': 'Cash Trgt',
    'EQTotal_Actl': 'EQ Actl', 'FITotal_Actl': 'FI Actl',
    'RETotal_Actl': 'RE Actl', 'PETotal_Actl': 'PE Actl',
    'HFTotal_Actl': 'HF Actl', 'CashTotal_Actl': 'Cash Actl',
    'ActFundedRatio_GASB': 'Funding Ratio',
    'InvestmentReturnAssumption_GASB': 'Hurdle Rate',
    'InvestmentReturn_1yr': 'Return 1yr',
    'ret_bgnassets': 'Net Return',
    'TotMembership': 'Tot Members',
    'actives_tot': 'Actives',
    'beneficiaries_tot': 'Retirees',
    'ActiveAge_avg': 'Active Age',
    'ActiveSalary_avg': 'Avg Salary',
    'BeneficiaryBenefit_avg': 'Avg Benefit',
    'MktAssets_net': 'AUM',
    'payroll': 'Payroll',
    'contrib_tot': 'Contributions',
}

corr = df[corr_cols].corr()
corr.index = [short_labels.get(c, c) for c in corr.index]
corr.columns = [short_labels.get(c, c) for c in corr.columns]

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Correlation Matrix — Key Variables', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'correlation_matrix.png')
plt.show()

# Print strongest correlations (excluding trivially related pairs)
print('\nStrongest correlations (|r| > 0.5, excluding same-variable pairs):')
corr_flat = corr.where(~mask & (corr.abs() > 0.5)).stack().reset_index()
corr_flat.columns = ['Var1', 'Var2', 'r']
corr_flat = corr_flat[corr_flat['Var1'] != corr_flat['Var2']]
corr_flat = corr_flat.sort_values('r', key=abs, ascending=False)
print(corr_flat.head(20).to_string(index=False))

## Section 6: Plan-Level vs. System-Level Structure

How many plans map to each system? How concentrated is AUM?

In [ ]:
# Plans per system
plans_per_system = df.groupby('pub_id')['ppd_id'].nunique().reset_index()
plans_per_system.columns = ['pub_id', 'n_plans']
plans_per_system = plans_per_system.merge(
    df.groupby('pub_id')['pub_system_name'].first().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of plans per system
ax = axes[0]
ax.hist(plans_per_system['n_plans'], bins=range(1, plans_per_system['n_plans'].max() + 2),
        color='#1f77b4', edgecolor='white', alpha=0.85, align='left')
ax.set_xlabel('Number of Plans per System')
ax.set_ylabel('Number of Systems')
ax.set_title(f'Plans per System (median={plans_per_system["n_plans"].median():.0f})')
ax.grid(axis='y', alpha=0.3)

# Right: AUM concentration
# Use most recent year for AUM
latest = df[df['fy'] == df['fy'].max()].copy()
sys_aum = latest.groupby('pub_id')['MktAssets_net'].sum().sort_values(ascending=False).reset_index()
sys_aum = sys_aum.merge(latest.groupby('pub_id')['pub_system_name'].first().reset_index())
sys_aum['cum_share'] = sys_aum['MktAssets_net'].cumsum() / sys_aum['MktAssets_net'].sum() * 100
sys_aum['rank'] = range(1, len(sys_aum) + 1)

ax = axes[1]
ax.plot(sys_aum['rank'], sys_aum['cum_share'], lw=2, color='#1f77b4')
ax.axhline(50, color='red', ls='--', alpha=0.5, lw=1)
ax.axhline(80, color='red', ls='--', alpha=0.5, lw=1)
# Mark top 10, 20
for n in [10, 20]:
    val = sys_aum.loc[sys_aum['rank'] == n, 'cum_share'].values[0]
    ax.annotate(f'Top {n}: {val:.0f}%', xy=(n, val),
                xytext=(n + 10, val - 5), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('System Rank (by AUM)')
ax.set_ylabel('Cumulative AUM Share (%)')
ax.set_title(f'AUM Concentration (FY{int(df["fy"].max())})')
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'plan_system_structure.png')
plt.show()

# Top 10 systems by AUM
print(f'\nTop 10 Systems by AUM (FY{int(df["fy"].max())}):')
top10 = sys_aum.head(10)[['rank', 'pub_system_name', 'MktAssets_net', 'cum_share']].copy()
top10['MktAssets_net'] = (top10['MktAssets_net'] / 1e9).round(1)
top10.columns = ['Rank', 'System', 'AUM ($B)', 'Cum Share (%)']
print(top10.to_string(index=False))

# Multi-plan systems
multi = plans_per_system[plans_per_system['n_plans'] > 1].sort_values('n_plans', ascending=False)
print(f'\nSystems with multiple plans: {len(multi)} / {len(plans_per_system)}')
print(multi.head(10)[['pub_system_name', 'n_plans']].to_string(index=False))